In [ ]:
# Exploratory CellOracle GRN and PAX8 knockout simulation (not part of the thesis).
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import celloracle as co
from celloracle.applications import Pseudotime_calculator
from celloracle.applications import Gradient_calculator
from celloracle.applications import Oracle_development_module

In [ ]:
%matplotlib inline
import scvelo as scv

In [ ]:
# Load data
adata = sc.read('/home/yanjun2/scanpy/8_figures/inte_palantir.h5ad')

In [ ]:
adata 

In [ ]:
sc.pp.highly_variable_genes(adata, subset = True, inplace = True)

In [ ]:
adata

In [ ]:
adata.obs['clusters'] = adata.obs['leiden_res_0_8']

In [ ]:
sc.pl.umap(adata, color='leiden_res_0_8', legend_loc='on data')

In [ ]:
scv.pl.diffmap(adata, color  = 'leiden_res_0_8')

In [ ]:
lineage_dict = {'the_one':adata.obs.leiden_res_0_8.unique().tolist()}
lineage_dict

In [ ]:
base_GRN = co.data.load_human_promoter_base_GRN()

In [ ]:
del adata.layers['data']
del adata.layers['scaled']
adata.X = adata.layers['counts'].copy()
oracle = co.Oracle()

In [ ]:
#oracle.import_anndata_as_raw_count(adata=adata,
 #                                  cluster_column_name="leiden_res_0_8",
  #                                 embedding_name="X_umap")

In [ ]:
oracle.import_anndata_as_raw_count(adata=adata,
                                   cluster_column_name="condition",
                                   embedding_name="X_umap")

In [ ]:
oracle.import_TF_data(TF_info_matrix=base_GRN)

In [ ]:
oracle.perform_PCA()

In [ ]:

# Select important PCs
plt.plot(np.cumsum(oracle.pca.explained_variance_ratio_)[:100])
n_comps = np.where(np.diff(np.diff(np.cumsum(oracle.pca.explained_variance_ratio_))>0.002))[0][0]
plt.axvline(n_comps, c="k")
plt.show()
print(n_comps)
n_comps = min(n_comps, 50)

In [ ]:
n_cell = oracle.adata.shape[0]
print(f"cell number is :{n_cell}")

In [ ]:

k = int(0.025*n_cell)
print(f"Auto-selected k is :{k}")

In [ ]:
oracle.knn_imputation(n_pca_dims=n_comps, k=k, balanced=True, b_sight=k*8,
                      b_maxl=k*4, n_jobs=12)


In [ ]:

%%time
links = oracle.get_links(cluster_name_for_GRN_unit="condition", alpha=10,
                         verbose_level=10)

In [ ]:
links.filter_links(p=0.001, weight="coef_abs", threshold_number=10000) #10k is default top num to keep

In [ ]:
links.get_network_score()

In [ ]:
links.merged_score

In [ ]:
links.plot_scores_as_rank(cluster="Mac4", n_gene=30)

In [ ]:

links.plot_scores_as_rank(cluster="MDIX", n_gene=30)

In [ ]:
links.plot_score_comparison_2D(value="eigenvector_centrality",
                               cluster1="Mac4", cluster2="Mac7",
                               percentile=98)

In [ ]:
links.plot_score_comparison_2D(value="eigenvector_centrality",
                               cluster1="MDIX", cluster2="MDI",
                               percentile=98)

In [ ]:
links.plot_score_per_cluster(goi="PAX8")

In [ ]:
oracle.get_cluster_specific_TFdict_from_Links(links_object=links)
oracle.fit_GRN_for_simulation(alpha=10, use_cluster_specific_TFdict=True)


In [ ]:
sc.pl.umap(oracle.adata, color = ['PAX8'], layer="imputed_count")


In [ ]:
oracle.simulate_shift(perturb_condition={'PAX8': 0.0}, n_propagation=2)

In [ ]:
oracle.estimate_transition_prob(n_neighbors=200,
                                knn_random=True,
                                sampled_fraction=1)

oracle.calculate_embedding_shift(sigma_corr=0.05)

In [ ]:
fig, ax = plt.subplots(1, 2,  figsize=[13, 6])

scale = 10#bigger = smaller vector
# Show quiver plot
oracle.plot_quiver(scale=scale, ax=ax[0])
ax[0].set_title(f"Simulated cell identity shift vector: PAX8 KO")

# Show quiver plot that was calculated with randomized graph.
oracle.plot_quiver_random(scale=scale, ax=ax[1])
ax[1].set_title(f"Randomized simulation vector")

plt.show()

In [ ]:
n_grid = 40
oracle.calculate_p_mass(smooth=0.8, n_grid=n_grid, n_neighbors=200)

In [ ]:
oracle.suggest_mass_thresholds(n_suggestion=12)

In [ ]:
min_mass = 9
oracle.calculate_mass_filter(min_mass=min_mass, plot=True)

In [ ]:
fig, ax = plt.subplots(1, 2,  figsize=[13, 6])

scale_simulation = 30
# Show quiver plot
oracle.plot_simulation_flow_on_grid(scale=scale_simulation, ax=ax[0])
ax[0].set_title(f"Simulated cell identity shift vector: PAX KO")

# Show quiver plot that was calculated with randomized graph.
oracle.plot_simulation_flow_random_on_grid(scale=scale_simulation, ax=ax[1])
ax[1].set_title(f"Randomized simulation vector")

plt.show()


In [ ]:
adata

In [ ]:


oracle.adata.obs['Pseudotime'] = adata.obs['palantir_pseudotime']


In [ ]:
gradient = Gradient_calculator(oracle_object=oracle, pseudotime_key="Pseudotime")

In [ ]:
gradient.calculate_p_mass(smooth=0.8, n_grid=n_grid, n_neighbors=200)
gradient.calculate_mass_filter(min_mass=min_mass, plot=False)

In [ ]:
gradient.transfer_data_into_grid(args={"method": "polynomial", "n_poly":3}, plot=True)

In [ ]:


gradient.calculate_gradient()



In [ ]:
scale_dev = 40
gradient.visualize_results(scale=scale_dev, s=5)

In [ ]:
fig, ax = plt.subplots(figsize=[6, 6])
gradient.plot_dev_flow_on_grid(scale=scale_dev, ax=ax)

In [ ]:
dev = Oracle_development_module()

In [ ]:
dev.load_differentiation_reference_data(gradient_object=gradient)

In [ ]:
dev.load_perturb_simulation_data(oracle_object=oracle)

In [ ]:
dev.calculate_inner_product()
dev.calculate_digitized_ip(n_bins=10)

In [ ]:
vm = 1 #adjust based on data

fig, ax = plt.subplots(1, 2, figsize=[12, 6])
dev.plot_inner_product_on_grid(vm=vm, s=50, ax=ax[0])
ax[0].set_title(f"PS")

dev.plot_inner_product_random_on_grid(vm=vm, s=50, ax=ax[1])
ax[1].set_title(f"PS calculated with Randomized simulation vector")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=[6, 6])
dev.plot_inner_product_on_grid(vm=vm, s=50, ax=ax)
dev.plot_simulation_flow_on_grid(scale=scale_simulation, show_background=False, ax=ax)

#plt.savefig('pertubation_score.png', dpi = 300)